In [5]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [8]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import load_dataset
import tqdm

tokenizer = AutoTokenizer.from_pretrained("JeremiahZ/roberta-base-qnli")
model = AutoModelForSequenceClassification.from_pretrained("JeremiahZ/roberta-base-qnli")

In [9]:
print(model)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [15]:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device='cuda')

In [ ]:
# two examples from training set

question = "When did the third Digimon series begin?"
sentence = "Unlike the two seasons before it and most of the seasons that followed, Digimon Tamers takes a darker and more realistic approach to its story featuring Digimon who do not reincarnate after their deaths and more complex character development in the original Japanese."

print(classifier(question+" "+sentence))

question = "What two things does Popper argue Tarski's theory involves in an evaluation of truth?"
sentence = "He bases this interpretation on the fact that examples such as the one described above refer to two things: assertions and the facts to which they refer."
print(classifier(question+" "+sentence))

[{'label': 'not_entailment', 'score': 0.9956406354904175}]
[{'label': 'entailment', 'score': 0.9965680837631226}]


In [24]:
dataset = load_dataset("glue", "qnli")['validation']
print(dataset)

Dataset({
    features: ['question', 'sentence', 'label', 'idx'],
    num_rows: 5463
})


In [28]:
correct = 0
total = 0

for i in tqdm.tqdm(range(5463)):
    result = classifier(dataset[i]['question'] + " " + dataset[i]['sentence'])
    result = 0 if result[0]['label'] == 'entailment' else 1
    if result == dataset[i]['label']:
        correct += 1
    total += 1

print("correct:{}, total:{}, accuracy:{}".format(correct, total, correct/total))

100%|██████████| 5463/5463 [00:32<00:00, 167.23it/s]

correct:5019, total:5463, accuracy:0.9187259747391543
